In [2]:
with open("data/input.txt", "r", encoding="utf-8") as f:
    text = f.read()

In [3]:
print("length of the dataset of characters:", len(text))

length of the dataset of characters: 1115394


In [4]:
chars = sorted(list(set(text)))
vocab_size = len(chars)
print(''.join(chars))
print(vocab_size)


 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz
65


In [5]:
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for i, ch in enumerate(chars)}
encoder = lambda s: [stoi[c] for c in s]
decoder = lambda l: ''.join([itos[i] for i in l])

print(encoder("hello"))
print(decoder(encoder("hello")))

[46, 43, 50, 50, 53]
hello


In [6]:
import torch 
data = torch.tensor(encoder(text),dtype= torch.long)


In [7]:
n = int(0.9 * len(data))
train_data = data[:n]   
val_data = data[n:]

In [66]:
import torch.nn as nn
import torch.nn.functional as F
import torch
block_size = 64
batch_size = 256
max_iters = 5000
eval_intervals = 500
learning_rate = 3e-3
device= 'cuda' if torch.cuda.is_available() else 'cpu'
eval_iters = 200
n_embd= 384
n_head = 6
n_layer = 6
dropout = 0.2

def get_batch(split):
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    return x.to(device), y.to(device)

In [ ]:
class Head(nn.Module) :
    """ one head of self attention"""
    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))
        self.dropout = nn.Dropout(dropout)
    def forward(self, x):
        B, T, C = x.shape
        k = self.key(x)   # (B, T, head_size)
        q = self.query(x) # (B, T, head_size)
        v = self.value(x) # (B, T, head_size)

        att = q @ k.transpose(-2, -1) * C**-0.5 # (B,T,C) @ (B, C, T) -> (B,T,T)
        att = att.masked_fill(self.tril[:T, :T] == 0, float('-inf'))
        att = torch.softmax(att, dim=-1)
        att = self.dropout(att)
        y = att @ v # (B, T, head_size)
        return y

class MultiHeadAttention(nn.Module):
    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(n_embd, n_embd)
        self.dropout = nn.Dropout(dropout)
    def forward(self, x):
        x = torch.cat([h(x) for h in self.heads], dim=-1)
        x = self.dropout(self.proj(x))
        return x

class FeedForward(nn.Module):
    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.ReLU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout)
        )

    def forward(self, x):
        return self.net(x)

class Block(nn.Module) : 
    """Transformer Block"""
    def __init__(self, n_embd, n_head):
        super().__init__()
        head_size=n_embd//n_head
        self.sa_heads = MultiHeadAttention(num_heads=n_head, head_size=head_size)
        self.ffwd = FeedForward(n_embd)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)
    def forward(self, x):
        x = x + self.sa_heads(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x

class BigramLanguageModel(nn.Module):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        self.blocks = nn.Sequential( *[Block(n_embd, n_head) for _ in range(n_layer)] )
        self.ln_f = nn.LayerNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size)
    
    def forward(self, idx, targets=None):
        #idx is initially of shape (B, T)
        tok_emb = self.token_embedding_table(idx) #(B,T,C)
        pos_emb = self.position_embedding_table(torch.arange(idx.shape[1], device=idx.device)) #(T,C)
        x = tok_emb + pos_emb # (B,T,C)
        x = self.blocks(x)
        x = self.ln_f(x)
        logits = self.lm_head(x)

        if targets is None:
            loss = None
        else : 
            B,T,C = logits.shape
            
            
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_new_tokens : int):
        for _ in range(max_new_tokens):
            #idx is of shape (B, T) however during generation T need not be the block size
            idx_cond = idx[:,-block_size:] # last block size length tokens used since we only use position embedding of the last block size length tokens to predict the next ones
            logits, loss = self(idx_cond)
            logits = logits[:, -1, :] # we only care about the last token
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)
        return idx

model = BigramLanguageModel()
model = model.to(device)

In [60]:
@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

In [64]:
print(sum(p.numel() for p in model.parameters()))

42369


In [65]:
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)
for step in range(max_iters):
    if step % eval_intervals == 0:
        losses = estimate_loss()
        print(f"step {step}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")
    X,Y = get_batch('train')
    logits,loss = model(X, Y)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

step 0: train loss 4.4818, val loss 4.4824
step 500: train loss 2.4539, val loss 2.4645
step 1000: train loss 2.3028, val loss 2.3213
step 1500: train loss 2.2346, val loss 2.2542
step 2000: train loss 2.1759, val loss 2.2153
step 2500: train loss 2.1539, val loss 2.1835
step 3000: train loss 2.1131, val loss 2.1597
step 3500: train loss 2.1242, val loss 2.1413
step 4000: train loss 2.0863, val loss 2.1265
step 4500: train loss 2.0670, val loss 2.1185


In [54]:
context = torch.zeros((1, 1), dtype=torch.long, device=device)
print(decoder(model.generate(context, max_new_tokens=10000)[0].tolist()))


To care you tim ent throunce took,;'s made eponmensbuting thy your pior sinconds lond than his is livip.

ETVLWARY It, I cond, but ig of a neck thout to blow his with voff much lurse owerry
he how to mucle, whith the slack keen bet. Lramewnsmansy prove enre bage base hot rewen will a spouattuties his all. Enow of this gearcouron, to good low, to shands I of mess thal an, wiwes, rease I
rewn to frupaurt prehteri', fir lorves but the meens, to monss? I-k I bidree.
But lighor gumbang it side,
If sircobly derlenst, side to him, I ssaut fir stenser, my mathern of my turn atth oly doot, have rust Warking some that past noge his at in the we with king hing loveir ing:
Thou to make to hot with our tad Edow.

KEeas bellis ous sepon ale twille! bortor?

AMLARGE:
Seclanw is the his?
In litnather I fittor,
And, I king if me cold age it:
Tere grand friges, yet be will fair, I poor cont: as wnow, with comeswen I,
'e my my sween
Mysere
Annot have as ame the said to; Deidy au fir I peak,
At state.

U